# SSL for Fundus Image Classification — Demo**End-to-end Self-Supervised Learning pipeline for multi-label disease classification on the ODIR dataset.****Author**: Sanjukta BiswasThis notebook implements:- **2 SSL pretraining methods**: SimCLR, BYOL- **ResNet-50** backbone- **Multi-label classification** for 4 ocular diseases- **Ensemble fusion** combining both methods**ODIR Disease Classes**: Normal, Diabetes, Glaucoma, Hypertension---**Instructions**: Run cells sequentially. Make sure GPU runtime is enabled:`Runtime → Change runtime type → T4 GPU`

## 0. Setup & Install Dependencies

In [ ]:
# Check GPU availability!nvidia-smiimport torchprint(f"\nPyTorch: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install any missing packages!pip install -q scikit-learn openpyxl

## 1. Download & Prepare ODIR DatasetThe **ODIR-5K** (Ocular Disease Intelligent Recognition) dataset contains5,000 patient records with fundus images of both eyes and diagnostic labels.### Option A: Kaggle API download (recommended)1. Download `kaggle.json` from [Kaggle Settings](https://www.kaggle.com/account)2. Upload it to this notebook (Files panel)3. Run the cell below

In [ ]:
# ── Option A: Kaggle API download ──import subprocess, ostry:    os.makedirs('~/.kaggle', exist_ok=True)    if os.path.exists('kaggle.json'):        subprocess.run(['cp', 'kaggle.json', os.path.expanduser('~/.kaggle/')], check=True)        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)        print("Kaggle credentials configured.")            subprocess.run(['kaggle', 'datasets', 'download', '-d', 'junior7/odir5k'], check=True)    subprocess.run(['unzip', '-q', 'odir5k.zip'], check=True)    print("ODIR dataset downloaded and extracted.")except Exception as e:    print(f"Kaggle download failed ({e}). Proceed manually or use Option B below.")

In [ ]:
import osfrom pathlib import Path# Set DATA_ROOT to your ODIR directoryDATA_ROOT = "/content/ODIR-5K_1"  # Adjust if differentos.makedirs(DATA_ROOT, exist_ok=True)print(f"DATA_ROOT: {DATA_ROOT}")print(f"Contents: {os.listdir(DATA_ROOT)[:10]}"

## 2. Configuration

In [ ]:
from dataclasses import dataclass, fieldfrom typing import List, Optional# ──────────────────────────────────────────# ODIR Disease Labels (4 categories)# ──────────────────────────────────────────DISEASE_LABELS = [    "Normal",    "Diabetes",    "Glaucoma",    "Hypertension",]NUM_CLASSES = len(DISEASE_LABELS)@dataclassclass Config:    """All-in-one configuration."""    # Data    data_root: str = DATA_ROOT    image_size: int = 224    num_workers: int = 2       # Colab has 2 CPUs    train_ratio: float = 0.7    val_ratio: float = 0.15    seed: int = 42    # Backbone    backbone_name: str = "resnet50"    # SimCLR    simclr_projection_dim: int = 128    simclr_hidden_dim: int = 2048    simclr_temperature: float = 0.07    # BYOL    byol_projection_dim: int = 256    byol_hidden_dim: int = 4096    byol_prediction_dim: int = 256    byol_ema_decay: float = 0.996    byol_ema_decay_end: float = 1.0    # Training    device: str = "cuda" if torch.cuda.is_available() else "cpu"    ssl_batch_size: int = 128    ssl_lr: float = 1e-3    ssl_weight_decay: float = 1e-6    ssl_warmup_epochs: int = 10    ssl_epochs: int = 50    ssl_momentum: float = 0.9    gradient_clip: float = 1.0    mixed_precision: bool = True    # Fine-tuning    ft_batch_size: int = 32    ft_lr: float = 1e-3    ft_weight_decay: float = 1e-6    ft_warmup_epochs: int = 5    ft_epochs: int = 60    ft_freeze_epochs: int = 10    ft_dropout: float = 0.5    ft_threshold: float = 0.5    ft_bce_weight: float = 0.6    ft_dice_weight: float = 0.4    ft_label_smoothing: float = 0.1    ft_early_stopping_patience: int = 10cfg = Config()print(f"Config initialized: {NUM_CLASSES} classes, {cfg.device} device")

## 3. Data Augmentations (Fundus-Aware)

In [ ]:
from torchvision import transformsimport torchvision.transforms.functional as TFimport randomimport torch.nn.functional as F# ──────────────────────────────────────────# SSL Augmentations (stronger)# ──────────────────────────────────────────class SimCLRAugmentation:    """SimCLR augmentation pipeline."""    def __init__(self, image_size=224):        self.transform = transforms.Compose([            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),            transforms.RandomHorizontalFlip(),            transforms.RandomVerticalFlip(),            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),            transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),            transforms.ToTensor(),            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),        ])        def __call__(self, x):        return self.transform(x), self.transform(x)class BYOLAugmentation:    """BYOL augmentation pipeline (asymmetric)."""    def __init__(self, image_size=224):        self.view1 = transforms.Compose([            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),            transforms.RandomHorizontalFlip(),            transforms.RandomVerticalFlip(),            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),            transforms.ToTensor(),            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),        ])        self.view2 = transforms.Compose([            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),            transforms.RandomHorizontalFlip(),            transforms.RandomVerticalFlip(),            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),            transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),            transforms.ToTensor(),            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),        ])        def __call__(self, x):        return self.view1(x), self.view2(x)# ──────────────────────────────────────────# Fine-tuning Augmentations (lighter)# ──────────────────────────────────────────class FinetuneAugmentation:    """Augmentation for fine-tuning."""    def __init__(self, image_size=224, is_train=True):        if is_train:            self.transform = transforms.Compose([                transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0)),                transforms.RandomHorizontalFlip(),                transforms.RandomVerticalFlip(),                transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),                transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),                transforms.ToTensor(),                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),            ])        else:            self.transform = transforms.Compose([                transforms.CenterCrop(image_size),                transforms.ToTensor(),                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),            ])        def __call__(self, x):        return self.transform(x)print("Augmentations initialized.")

## 4. ODIR Dataset Loader

In [ ]:
import numpy as npimport pandas as pdfrom pathlib import Pathfrom PIL import Imagefrom torch.utils.data import Dataset, DataLoaderfrom sklearn.model_selection import train_test_splitclass ODIRDataset(Dataset):    """    ODIR fundus dataset.    Modes: 'ssl' (returns augmented views) or 'finetune' (returns image + labels).    """    def __init__(self, data_root, transform=None, mode="finetune"):        self.data_root = Path(data_root)        self.transform = transform        self.mode = mode        self.samples = self._load_samples()    def _load_samples(self):        samples = []        # Find annotation file        csvs = list(self.data_root.glob("*.csv")) + list(self.data_root.glob("*.xlsx"))        if not csvs:            # No annotations — load all images with empty labels (SSL mode)            img_dir = self._find_img_dir()            for f in sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png")):                samples.append({"path": str(f), "labels": np.zeros(NUM_CLASSES, dtype=np.float32)})            print(f"[DATA] Loaded {len(samples)} images (no annotations — SSL mode)")            return samples        ann_file = csvs[0]        df = pd.read_excel(ann_file) if str(ann_file).endswith('.xlsx') else pd.read_csv(ann_file)        img_dir = self._find_img_dir()        label_cols = self._detect_label_cols(df)        for _, row in df.iterrows():            for eye in ["left", "right"]:                path = self._resolve_path(row, eye, img_dir)                if path and path.exists():                    labels = self._extract_labels(row, label_cols)                    samples.append({"path": str(path), "labels": labels})        print(f"[DATA] Loaded {len(samples)} fundus images")        return samples    def _find_img_dir(self):        """Recursively find image directory."""        for d in [self.data_root / "ODIR-5K_Training_Data",                  self.data_root / "Training Images",                  self.data_root]:            if d.exists():                return d        raise FileNotFoundError(f"Image directory not found in {self.data_root}")    def _resolve_path(self, row, eye, img_dir):        """Find image path from row metadata."""        # Try multiple filename patterns        basename = row.get("ID") or row.get("Patient ID") or "unknown"        for ext in [".jpg", ".png", ".jpeg"]:            for pattern in [f"{basename}_{eye[0].upper()}{ext}",                           f"{basename}_{eye.upper()}{ext}",                           f"{basename}_{eye}{ext}"]:                p = img_dir / pattern                if p.exists():                    return p        return None    def _detect_label_cols(self, df):        """Detect which columns contain disease labels."""        # Look for columns with pattern of 0/1 values        selected_cols = ["N", "D", "G", "H"]  # 4-class mapping        found = [c for c in selected_cols if c in df.columns]        if found:            return found        # Fallback: look for numeric columns        return [c for c in df.columns if df[c].dtype in [int, float] and df[c].nunique() <= 2][:NUM_CLASSES]    def _extract_labels(self, row, label_cols):        """Extract multi-label binary vector."""        labels = np.zeros(NUM_CLASSES, dtype=np.float32)        # Map columns: N->0, D->1, G->2, H->3        col_map = {"N": 0, "D": 1, "G": 2, "H": 3}        for col in label_cols:            if col in col_map:                idx = col_map[col]                labels[idx] = float(row.get(col, 0))                # Keyword-based detection for missing labels        keyword_map = {            0: ["normal"],            1: ["diabetes", "diabetic", "retinopathy", "nonproliferative", "proliferative", "macular edema"],            2: ["glaucoma", "optic", "cup"],            3: ["hypertension", "hypertensive"],        }                note = str(row.get("Diagnostic Keywords", "")).lower()        for class_idx, keywords in keyword_map.items():            if any(kw in note for kw in keywords):                labels[class_idx] = 1.0                return labels    def __len__(self):        return len(self.samples)    def __getitem__(self, idx):        sample = self.samples[idx]        try:            img = Image.open(sample["path"]).convert("RGB")        except:            img = Image.new("RGB", (224, 224))                if self.mode == "ssl":            return self.transform(img)        else:  # finetune            img = self.transform(img)            return img, torch.tensor(sample["labels"], dtype=torch.float32)def get_dataloaders(mode="finetune", train_transform=None, val_transform=None, ssl_transform=None):    """Create train/val/test dataloaders."""    if mode == "ssl":        ds = ODIRDataset(cfg.data_root, transform=ssl_transform, mode="ssl")        train_size = int(0.9 * len(ds))        train_ds, val_ds = torch.utils.data.random_split(ds, [train_size, len(ds) - train_size])        return {            "train": DataLoader(train_ds, batch_size=cfg.ssl_batch_size, shuffle=True,                               num_workers=cfg.num_workers, pin_memory=True, drop_last=True),            "val": DataLoader(val_ds, batch_size=cfg.ssl_batch_size, shuffle=False,                             num_workers=cfg.num_workers, pin_memory=True),        }    else:  # finetune        ds = ODIRDataset(cfg.data_root, transform=None, mode="finetune")                # Manual split: train/val/test        n = len(ds)        indices = np.arange(n)        np.random.seed(cfg.seed)        np.random.shuffle(indices)                train_n = int(cfg.train_ratio * n)        val_n = int(cfg.val_ratio * n)                train_idx = indices[:train_n]        val_idx = indices[train_n:train_n + val_n]        test_idx = indices[train_n + val_n:]                from torch.utils.data import Subset        train_ds = Subset(ds, train_idx)        val_ds = Subset(ds, val_idx)        test_ds = Subset(ds, test_idx)                # Apply transforms        for subset in [train_ds, val_ds, test_ds]:            for i in subset.indices:                ds.samples[i]['transform'] = train_transform if subset is train_ds else val_transform                # Wrapper to apply per-sample transforms        class TransformedSubset(Subset):            def __getitem__(self, idx):                img, lbl = super().__getitem__(idx)                transform = train_transform if self is train_ds else val_transform                return transform(img) if transform else img, lbl                train_ds = TransformedSubset(ds, train_idx)        val_ds = TransformedSubset(ds, val_idx)        test_ds = TransformedSubset(ds, test_idx)                return {            "train": DataLoader(train_ds, batch_size=cfg.ft_batch_size, shuffle=True,                               num_workers=cfg.num_workers, pin_memory=True),            "val": DataLoader(val_ds, batch_size=cfg.ft_batch_size, shuffle=False,                             num_workers=cfg.num_workers, pin_memory=True),            "test": DataLoader(test_ds, batch_size=cfg.ft_batch_size, shuffle=False,                              num_workers=cfg.num_workers, pin_memory=True),            "full_dataset": ds,        }import torchprint("Dataset loader initialized.")

## 5. Backbone & Models

In [ ]:
import torch.nn as nnimport torchvision.models as modelsdef get_backbone(name="resnet50"):    """Get pretrained backbone."""    model = models.resnet50(pretrained=True)    model.fc = nn.Identity()    model.feature_dim = 2048    return modelclass MultiLabelClassifier(nn.Module):    """Multi-label classifier head."""    def __init__(self, backbone, num_classes=NUM_CLASSES, dropout=0.5, freeze=False):        super().__init__()        self.backbone = backbone        self.freeze_backbone = freeze        if freeze:            for p in backbone.parameters():                p.requires_grad = False        self.head = nn.Sequential(            nn.Linear(backbone.feature_dim, 512), nn.ReLU(True), nn.Dropout(dropout),            nn.Linear(512, num_classes),        )        def forward(self, x):        feat = self.backbone(x)        return self.head(feat)        def unfreeze(self):        for p in self.backbone.parameters():            p.requires_grad = Trueprint("Backbone models initialized.")

## 6. SSL Methods: SimCLR, BYOL

In [ ]:
import copyimport mathimport torch.nn.functional as F# ══════════════════════════════════════════# SimCLR# ══════════════════════════════════════════class SimCLR(nn.Module):    def __init__(self, backbone_name="resnet50", proj_dim=128, hidden_dim=2048, temperature=0.07):        super().__init__()        self.temperature = temperature        self.backbone = get_backbone(backbone_name)        fd = self.backbone.feature_dim        self.projector = nn.Sequential(            nn.Linear(fd, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(True),            nn.Linear(hidden_dim, proj_dim),        )    def forward(self, x1, x2):        z1 = F.normalize(self.projector(self.backbone(x1)), dim=1)        z2 = F.normalize(self.projector(self.backbone(x2)), dim=1)        return self._nt_xent(z1, z2), z1, z2    def _nt_xent(self, z1, z2):        B = z1.shape[0]        z = torch.cat([z1, z2], dim=0)        sim = torch.mm(z, z.T) / self.temperature        mask = torch.eye(2*B, device=z.device, dtype=torch.bool)        sim.masked_fill_(mask, float('-inf'))        pos_idx = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)        pos_sim = sim[torch.arange(2*B, device=z.device), pos_idx]        return (-pos_sim + torch.logsumexp(sim, dim=1)).mean()# ══════════════════════════════════════════# BYOL# ══════════════════════════════════════════class BYOL(nn.Module):    def __init__(self, backbone_name="resnet50", proj_dim=256, hidden_dim=4096,                 pred_dim=256, ema_decay=0.996):        super().__init__()        self.ema_decay = ema_decay                # Online network        self.online_backbone = get_backbone(backbone_name)        fd = self.online_backbone.feature_dim        self.online_projector = nn.Sequential(            nn.Linear(fd, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(True),            nn.Linear(hidden_dim, proj_dim), nn.BatchNorm1d(proj_dim),        )        self.predictor = nn.Sequential(            nn.Linear(proj_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(True),            nn.Linear(hidden_dim, proj_dim),        )                # Target network (EMA)        self.target_backbone = copy.deepcopy(self.online_backbone)        self.target_projector = copy.deepcopy(self.online_projector)                for p in self.target_backbone.parameters():            p.requires_grad = False        for p in self.target_projector.parameters():            p.requires_grad = False    def forward(self, x1, x2):        # Online        z1_online = self.online_projector(self.online_backbone(x1))        h1 = self.predictor(z1_online)                # Target        with torch.no_grad():            z2_target = self.target_projector(self.target_backbone(x2))                loss = 2 - 2 * F.cosine_similarity(h1, z2_target, dim=1).mean()                # Symmetry        z2_online = self.online_projector(self.online_backbone(x2))        h2 = self.predictor(z2_online)        with torch.no_grad():            z1_target = self.target_projector(self.target_backbone(x1))        loss = loss + (2 - 2 * F.cosine_similarity(h2, z1_target, dim=1).mean())                return loss / 2    def update_target(self, decay):        """Update target network via EMA."""        for p_online, p_target in zip(self.online_backbone.parameters(),                                        self.target_backbone.parameters()):            p_target.data = decay * p_target.data + (1 - decay) * p_online.data        for p_online, p_target in zip(self.online_projector.parameters(),                                       self.target_projector.parameters()):            p_target.data = decay * p_target.data + (1 - decay) * p_online.data    @staticmethod    def cosine_ema(start, end, step, total):        """Cosine schedule for EMA decay."""        return end + 0.5 * (start - end) * (1 + math.cos(math.pi * step / total))print("SSL methods (SimCLR, BYOL) initialized.")

## 7. Losses & Metrics

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_scoreclass CombinedLoss(nn.Module):    """BCE + Dice for multi-label classification."""    def __init__(self, bce_weight=0.6, dice_weight=0.4, label_smoothing=0.1, class_weights=None):        super().__init__()        self.bce_weight = bce_weight        self.dice_weight = dice_weight        self.bce = nn.BCEWithLogitsLoss(weight=class_weights, reduction='mean')        self.label_smoothing = label_smoothing        def forward(self, logits, targets):        # Label smoothing        targets_smooth = targets * (1 - self.label_smoothing) + self.label_smoothing / 2                bce_loss = self.bce(logits, targets_smooth)                # Dice loss        probs = torch.sigmoid(logits)        dice_loss = 1 - (2 * (probs * targets_smooth).sum(dim=0) + 1e-8) / ((probs + targets_smooth).sum(dim=0) + 1e-8)        dice_loss = dice_loss.mean()                return self.bce_weight * bce_loss + self.dice_weight * dice_lossdef compute_metrics(y_true, y_pred, y_prob):    """Compute multi-label metrics."""    metrics = {}        # Per-class AUC    auc_scores = []    for i in range(y_true.shape[1]):        if len(np.unique(y_true[:, i])) > 1:            auc = roc_auc_score(y_true[:, i], y_prob[:, i])            auc_scores.append(auc)    metrics['auc_macro'] = np.mean(auc_scores) if auc_scores else 0.0        # F1 (macro)    f1_scores = []    for i in range(y_true.shape[1]):        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)        f1_scores.append(f1)    metrics['f1_macro'] = np.mean(f1_scores)        # Precision, Recall    metrics['precision_macro'] = np.mean([precision_score(y_true[:, i], y_pred[:, i], zero_division=0) for i in range(y_true.shape[1])])    metrics['recall_macro'] = np.mean([recall_score(y_true[:, i], y_pred[:, i], zero_division=0) for i in range(y_true.shape[1])])        # Mean Average Precision    map_scores = []    for i in range(y_true.shape[1]):        if len(np.unique(y_true[:, i])) > 1:            ap = average_precision_score(y_true[:, i], y_prob[:, i])            map_scores.append(ap)    metrics['mAP'] = np.mean(map_scores) if map_scores else 0.0        # Exact match    metrics['exact_match'] = np.mean(np.all(y_true == y_pred, axis=1))        return metricsdef compute_class_weights(dataset):    """Compute class weights for imbalance."""    labels = np.array([s['labels'] for s in dataset.samples])    pos_counts = labels.sum(axis=0)    neg_counts = (1 - labels).sum(axis=0)    weights = (neg_counts + 1) / (pos_counts + 1)    return torch.tensor(weights, dtype=torch.float32)print("Loss and metrics initialized.")

## 8. Training Utilities

In [ ]:
from torch.cuda.amp import GradScaler, autocastimport timeclass EarlyStopping:    """Early stopping callback."""    def __init__(self, patience=10, min_delta=0):        self.patience = patience        self.min_delta = min_delta        self.counter = 0        self.best_loss = None        def __call__(self, val_loss):        if self.best_loss is None:            self.best_loss = val_loss        elif val_loss < self.best_loss - self.min_delta:            self.best_loss = val_loss            self.counter = 0        else:            self.counter += 1            if self.counter >= self.patience:                return True  # Stop        return Falsedef get_cosine_schedule(optimizer, warmup_epochs, total_epochs, num_steps_per_epoch=1):    """Cosine annealing with warmup."""    from torch.optim.lr_scheduler import LambdaLR    def lr_lambda(epoch):        if epoch < warmup_epochs:            return (epoch + 1) / warmup_epochs        return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (total_epochs - warmup_epochs)))    return LambdaLR(optimizer, lr_lambda)print("Training utilities initialized.")

## 9. SSL PretrainingNote: Only SimCLR (9a) and BYOL (9b) are used in this demo.

In [ ]:
# ──────────────────────────────────────# 9a. SimCLR Pretraining# ──────────────────────────────────────print("="*60)print("  SimCLR Pretraining")print("="*60)device = torch.device(cfg.device)aug = SimCLRAugmentation(cfg.image_size)loaders = get_dataloaders(mode="ssl", ssl_transform=aug)simclr = SimCLR(    proj_dim=cfg.simclr_projection_dim,    hidden_dim=cfg.simclr_hidden_dim,    temperature=cfg.simclr_temperature,).to(device)opt = torch.optim.AdamW(simclr.parameters(), lr=cfg.ssl_lr, weight_decay=cfg.ssl_weight_decay)sched = get_cosine_schedule(opt, cfg.ssl_warmup_epochs, cfg.ssl_epochs)scaler = GradScaler(enabled=cfg.mixed_precision)simclr_losses = []for epoch in range(cfg.ssl_epochs):    simclr.train()    total_loss, t0 = 0.0, time.time()    for v1, v2 in loaders["train"]:        v1, v2 = v1.to(device), v2.to(device)        opt.zero_grad()        with autocast(enabled=cfg.mixed_precision):            loss, _, _ = simclr(v1, v2)        scaler.scale(loss).backward()        scaler.unscale_(opt)        nn.utils.clip_grad_norm_(simclr.parameters(), cfg.gradient_clip)        scaler.step(opt)        scaler.update()        total_loss += loss.item()    sched.step()    avg = total_loss / len(loaders["train"])    simclr_losses.append(avg)    if (epoch+1) % 10 == 0 or epoch == 0:        print(f"  Epoch {epoch+1}/{cfg.ssl_epochs} | Loss: {avg:.4f} | {time.time()-t0:.1f}s")trained_backbones = {}trained_backbones["simclr"] = copy.deepcopy(simclr.backbone.state_dict())print(f"SimCLR training complete. Final loss: {simclr_losses[-1]:.4f}")

In [ ]:
# ──────────────────────────────────────# 9b. BYOL Pretraining# ──────────────────────────────────────print("="*60)print("  BYOL Pretraining")print("="*60)aug = BYOLAugmentation(cfg.image_size)loaders = get_dataloaders(mode="ssl", ssl_transform=aug)byol = BYOL(    proj_dim=cfg.byol_projection_dim,    hidden_dim=cfg.byol_hidden_dim,    pred_dim=cfg.byol_prediction_dim,    ema_decay=cfg.byol_ema_decay,).to(device)online_params = list(byol.online_backbone.parameters()) + list(byol.online_projector.parameters()) + list(byol.predictor.parameters())opt = torch.optim.AdamW(online_params, lr=cfg.ssl_lr, weight_decay=cfg.ssl_weight_decay)sched = get_cosine_schedule(opt, cfg.ssl_warmup_epochs, cfg.ssl_epochs)scaler = GradScaler(enabled=cfg.mixed_precision)total_steps = cfg.ssl_epochs * len(loaders["train"])step = 0byol_losses = []for epoch in range(cfg.ssl_epochs):    byol.train()    total_loss, t0 = 0.0, time.time()    for v1, v2 in loaders["train"]:        v1, v2 = v1.to(device), v2.to(device)        opt.zero_grad()        with autocast(enabled=cfg.mixed_precision):            loss = byol(v1, v2)        scaler.scale(loss).backward()        scaler.unscale_(opt)        nn.utils.clip_grad_norm_(online_params, cfg.gradient_clip)        scaler.step(opt)        scaler.update()        decay = BYOL.cosine_ema(cfg.byol_ema_decay, cfg.byol_ema_decay_end, step, total_steps)        byol.update_target(decay)        step += 1        total_loss += loss.item()    sched.step()    avg = total_loss / len(loaders["train"])    byol_losses.append(avg)    if (epoch+1) % 10 == 0 or epoch == 0:        print(f"  Epoch {epoch+1}/{cfg.ssl_epochs} | Loss: {avg:.4f} | {time.time()-t0:.1f}s")trained_backbones["byol"] = copy.deepcopy(byol.online_backbone.state_dict())print(f"BYOL training complete. Final loss: {byol_losses[-1]:.4f}")

In [ ]:
# ── Plot SSL pretraining losses ──import matplotlib.pyplot as pltfig, ax = plt.subplots(1, 1, figsize=(10, 5))ax.plot(simclr_losses, label="SimCLR", marker='o', markersize=3, linewidth=2)ax.plot(byol_losses, label="BYOL", marker='s', markersize=3, linewidth=2)ax.set_xlabel("Epoch", fontsize=12)ax.set_ylabel("Loss", fontsize=12)ax.set_title("SSL Pretraining Losses", fontsize=14, fontweight="bold")ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 10. Fine-Tuning & Evaluation

In [ ]:
def finetune_and_evaluate(method_name, backbone_state):    """    Fine-tune a pretrained backbone and evaluate on test set.    Returns: (model, test_metrics, y_true, y_prob, y_pred, history)    """    print(f"\n{'='*50}")    print(f"  Fine-tuning: {method_name.upper()}")    print(f"{'='*50}")    # Data    train_aug = FinetuneAugmentation(cfg.image_size, is_train=True)    val_aug = FinetuneAugmentation(cfg.image_size, is_train=False)    loaders = get_dataloaders(mode="finetune", train_transform=train_aug, val_transform=val_aug)    # Class weights    weights = compute_class_weights(loaders["full_dataset"]).to(device)    # Model    backbone = get_backbone()    backbone.load_state_dict(backbone_state)    model = MultiLabelClassifier(backbone, dropout=cfg.ft_dropout,                                  freeze=(cfg.ft_freeze_epochs > 0)).to(device)    criterion = CombinedLoss(cfg.ft_bce_weight, cfg.ft_dice_weight,                             cfg.ft_label_smoothing, weights)    # Optimizer (only head when frozen)    if cfg.ft_freeze_epochs > 0:        opt = torch.optim.AdamW(model.head.parameters(), lr=cfg.ft_lr, weight_decay=cfg.ft_weight_decay)    else:        opt = torch.optim.AdamW(model.parameters(), lr=cfg.ft_lr, weight_decay=cfg.ft_weight_decay)    sched = get_cosine_schedule(opt, cfg.ft_warmup_epochs, cfg.ft_epochs)    scaler = GradScaler(enabled=cfg.mixed_precision)    early = EarlyStopping(cfg.ft_early_stopping_patience)    history = {"train_loss": [], "val_loss": [], "val_auc": []}    for epoch in range(cfg.ft_epochs):        # Train        model.train()        total_loss = 0        for x, y in loaders["train"]:            x, y = x.to(device), y.to(device)            opt.zero_grad()            with autocast(enabled=cfg.mixed_precision):                logits = model(x)                loss = criterion(logits, y)            scaler.scale(loss).backward()            scaler.unscale_(opt)            nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)            scaler.step(opt)            scaler.update()            total_loss += loss.item()                # Validate        model.eval()        val_loss, y_val_true, y_val_prob = 0, [], []        with torch.no_grad():            for x, y in loaders["val"]:                x, y = x.to(device), y.to(device)                logits = model(x)                loss = criterion(logits, y)                val_loss += loss.item()                y_val_true.append(y.cpu().numpy())                y_val_prob.append(torch.sigmoid(logits).cpu().numpy())                y_val_true = np.concatenate(y_val_true)        y_val_prob = np.concatenate(y_val_prob)        val_metrics = compute_metrics(y_val_true, (y_val_prob >= cfg.ft_threshold).astype(float), y_val_prob)                avg_train_loss = total_loss / len(loaders["train"])        avg_val_loss = val_loss / len(loaders["val"])                history["train_loss"].append(avg_train_loss)        history["val_loss"].append(avg_val_loss)        history["val_auc"].append(val_metrics["auc_macro"])                if (epoch+1) % 10 == 0 or epoch == 0:            print(f"Epoch {epoch+1}/{cfg.ft_epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | AUC: {val_metrics['auc_macro']:.4f}")                sched.step()                # Unfreeze after freeze period        if epoch == cfg.ft_freeze_epochs - 1:            model.unfreeze()            opt = torch.optim.AdamW(model.parameters(), lr=cfg.ft_lr, weight_decay=cfg.ft_weight_decay)                if early(avg_val_loss):            print(f"Early stopping at epoch {epoch+1}")            break        # Evaluate on test set    model.eval()    y_test_true, y_test_prob = [], []    with torch.no_grad():        for x, y in loaders["test"]:            x, y = x.to(device), y.to(device)            logits = model(x)            y_test_true.append(y.cpu().numpy())            y_test_prob.append(torch.sigmoid(logits).cpu().numpy())        y_test_true = np.concatenate(y_test_true)    y_test_prob = np.concatenate(y_test_prob)    y_test_pred = (y_test_prob >= cfg.ft_threshold).astype(float)        test_metrics = compute_metrics(y_test_true, y_test_pred, y_test_prob)        print(f"\nTest Results:")    for k, v in test_metrics.items():        print(f"  {k}: {v:.4f}")        return model, test_metrics, y_test_true, y_test_prob, y_test_pred, history# Fine-tune all two methodsresults = {}for method_name, backbone_state in trained_backbones.items():    model, metrics, y_true, y_prob, y_pred, hist = finetune_and_evaluate(method_name, backbone_state)    results[method_name] = {        "model": model,        "metrics": metrics,        "y_true": y_true,        "y_prob": y_prob,        "y_pred": y_pred,        "history": hist,    }print("\n" + "="*50)print("Fine-tuning complete!")print("="*50)

## 11. Ensemble Fusion

In [ ]:
# ── Combine predictions from both methods ──from sklearn.metrics import roc_curve, auc as sk_auc, confusion_matrix# Get a common y_true (they're all from the same test split)y_true = list(results.values())[0]["y_true"]all_probs = {m: r["y_prob"] for m, r in results.items()}# Fusion strategiesdef fuse(probs, strategy="average", weights=None):    arrays = list(probs.values())    if strategy == "average":        return np.mean(arrays, axis=0)    elif strategy == "weighted":        total = sum(weights.values())        return sum(weights[m] / total * probs[m] for m in probs)    elif strategy == "max":        return np.maximum.reduce(arrays)# Use validation AUC as weightsfusion_weights = {m: r["metrics"]["auc_macro"] for m, r in results.items()}print("Ensemble Results:")print(f"{'Strategy':<12} {'AUC':>8} {'F1':>8} {'mAP':>8} {'Exact':>8}")print("-" * 48)ensemble_results = {}for strat in ["average", "weighted", "max"]:    fused = fuse(all_probs, strat, fusion_weights)    preds = (fused >= cfg.ft_threshold).astype(float)    m = compute_metrics(y_true, preds, fused)    ensemble_results[strat] = {"probs": fused, "preds": preds, "metrics": m}    print(f"{strat:<12} {m['auc_macro']:>8.4f} {m['f1_macro']:>8.4f} {m['mAP']:>8.4f} {m['exact_match']:>8.4f}")# Also show individual resultsprint("\nIndividual Method Results:")print(f"{'Method':<12} {'AUC':>8} {'F1':>8} {'mAP':>8} {'Exact':>8}")print("-" * 48)for method, r in results.items():    m = r["metrics"]    print(f"{method:<12} {m['auc_macro']:>8.4f} {m['f1_macro']:>8.4f} {m['mAP']:>8.4f} {m['exact_match']:>8.4f}")# Best ensemble strategybest_strat = max(ensemble_results, key=lambda k: ensemble_results[k]["metrics"]["auc_macro"])print(f"\nBest ensemble strategy: {best_strat}")

## 12. Visualization

In [ ]:
# ── ROC curves (ensemble) ──best_probs = ensemble_results[best_strat]["probs"]fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))for i, (label, ax) in enumerate(zip(DISEASE_LABELS, axes)):    fpr, tpr, _ = roc_curve(y_true[:, i], best_probs[:, i])    roc_auc = sk_auc(fpr, tpr)    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')    ax.set_title(label, fontweight='bold')    ax.legend(loc="lower right")    ax.grid(True, alpha=0.3)plt.suptitle(f"ROC Curves — Ensemble ({best_strat})", fontsize=14, y=1.02)plt.tight_layout(); plt.show()

In [ ]:
# ── Confusion matrices per class ──best_preds = ensemble_results[best_strat]["preds"]fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))for i, (label, ax) in enumerate(zip(DISEASE_LABELS, axes)):    cm = confusion_matrix(y_true[:, i], best_preds[:, i], labels=[0, 1])    ax.imshow(cm, cmap="Blues")    ax.set_title(label, fontweight="bold")    ax.set_xticks([0,1]); ax.set_yticks([0,1])    ax.set_xticklabels(["Neg", "Pos"]); ax.set_yticklabels(["Neg", "Pos"])    ax.set_xlabel("Predicted"); ax.set_ylabel("True")    for r in range(2):        for c in range(2):            ax.text(c, r, str(cm[r,c]), ha="center", va="center", fontsize=14,                    color="white" if cm[r,c] > cm.max()/2 else "black")plt.suptitle(f"Confusion Matrices — Ensemble ({best_strat})", fontsize=14, y=1.02)plt.tight_layout(); plt.show()

## 13. Save Models & Results

In [ ]:
import jsonimport os# Create output directoryoutput_dir = "/content/ssl_results"os.makedirs(output_dir, exist_ok=True)# Save metricsmetrics_summary = {    "individual": {m: r["metrics"] for m, r in results.items()},    "ensemble": {s: r["metrics"] for s, r in ensemble_results.items()},    "best_ensemble": best_strat,}with open(f"{output_dir}/metrics.json", "w") as f:    json.dump({k: {m: float(v) for m, v in vals.items()} if isinstance(vals, dict) else vals                for k, vals in metrics_summary.items()}, f, indent=2)print(f"Metrics saved to {output_dir}/metrics.json")# Save predictionspredictions = {    "y_true": y_true.tolist(),    "ensemble_probs": ensemble_results[best_strat]["probs"].tolist(),    "ensemble_preds": ensemble_results[best_strat]["preds"].tolist(),}with open(f"{output_dir}/predictions.json", "w") as f:    json.dump(predictions, f)print(f"Predictions saved to {output_dir}/predictions.json")# Save modelsfor method_name, r in results.items():    torch.save(r["model"].state_dict(), f"{output_dir}/{method_name}_model.pth")    print(f"Model saved: {output_dir}/{method_name}_model.pth")print(f"\nAll results saved to {output_dir}")

---## Quick Reference### Key hyperparameters to tune:| Parameter | Location | Default | Notes ||---|---|---|---|| `cfg.ssl_epochs` | Config cell | 50 | Increase for better features || `cfg.ssl_batch_size` | Config cell | 128 | Larger = better for SimCLR || `cfg.ft_epochs` | Config cell | 60 | Early stopping handles this || `cfg.ft_freeze_epochs` | Config cell | 10 | Epochs with frozen backbone || `cfg.ft_lr` | Config cell | 1e-3 | For classifier head || `cfg.simclr_temperature` | Config cell | 0.07 | Lower = sharper contrasts |### Common issues:- **Low AUC**: Train SSL longer, use larger batch for SimCLR- **Class imbalance**: Adjust class weights in `compute_class_weights()`- **Memory issues**: Reduce batch sizes or image size to 192### Methods used:- **SimCLR** (9a): Contrastive learning with NT-Xent loss- **BYOL** (9b): Bootstrap Your Own Latent (no negatives required)- **Ensemble**: Average, weighted, or max voting fusion